# MobiDB Gold dataset exploration

Notebook for opening `data/processed/mobidb/mobidb_gold_2022_07.json` without loading the whole file into memory. If the processed JSON is not ready yet, it can also preview the raw `data/raw/mobidb/mobidb_gold_2022_07.mjson.gz` archive.

In [29]:
from pathlib import Path
from collections import Counter
import gzip
import json
import itertools
import matplotlib.pyplot as plt

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = PROJECT_ROOT / "data/processed/mobidb/mobidb_gold_2022_07.json"
RAW_ARCHIVE_PATH = PROJECT_ROOT / "data/raw/mobidb/mobidb_gold_2022_07.mjson.gz"

print("processed:", DATASET_PATH)
print("raw:", RAW_ARCHIVE_PATH)

processed: /Users/apple/BMM/gen-ai/data/processed/mobidb/mobidb_gold_2022_07.json
raw: /Users/apple/BMM/gen-ai/data/raw/mobidb/mobidb_gold_2022_07.mjson.gz


In [30]:
def iter_processed_json_array(path: Path):
    """Yield records from the pipeline JSON array one object at a time."""
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line or line in {"[", "]", ","}:
                continue
            if line.endswith(","):
                line = line[:-1]
            yield json.loads(line)


def iter_raw_mjson_gz(path: Path):
    """Yield records from the original MobiDB gzip-compressed JSON Lines archive."""
    with gzip.open(path, mode="rt", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                yield json.loads(line)


def iter_mobidb_records():
    if DATASET_PATH.exists():
        print("opening processed JSON")
        return iter_processed_json_array(DATASET_PATH)
    if RAW_ARCHIVE_PATH.exists():
        print("processed JSON not found; opening raw .mjson.gz preview")
        return iter_raw_mjson_gz(RAW_ARCHIVE_PATH)
    raise FileNotFoundError(
        "Run `nextflow run pipeline/build_mobidb_gold_json.nf` first, "
        "or download the raw MobiDB archive."
    )

## Raw `.mjson.gz` preview

The source MobiDB archive is gzip-compressed JSON Lines: one full JSON object per line. These cells show the raw structure before the pipeline converts it to the compact dataset.

In [31]:
RAW_PREVIEW_SIZE = 3

if RAW_ARCHIVE_PATH.exists():
    raw_records = list(itertools.islice(iter_raw_mjson_gz(RAW_ARCHIVE_PATH), RAW_PREVIEW_SIZE))
    print("raw preview rows:", len(raw_records))

    raw_summary = []
    for record in raw_records:
        raw_summary.append(
            {
                "acc": record.get("acc"),
                "length": record.get("length"),
                "organism": record.get("organism"),
                "ncbi_taxon_id": record.get("ncbi_taxon_id"),
                "top_level_keys": len(record),
                "region_blocks": sum(isinstance(value, dict) and "regions" in value for value in record.values()),
                "score_blocks": sum(isinstance(value, dict) and "scores" in value for value in record.values()),
            }
        )

    display(pd.DataFrame(raw_summary))
    print("first raw record keys:")
    print(sorted(raw_records[0].keys()) if raw_records else [])
else:
    raw_records = []
    print("raw archive not found:", RAW_ARCHIVE_PATH)

raw preview rows: 3


,acc,length,organism,ncbi_taxon_id,top_level_keys,region_blocks,score_blocks
0,W6PPR4,4332,Mus musculus (Mouse),10090,49,28,8
1,Q14596,966,Homo sapiens (Human),9606,87,55,19
2,Q96CV9,577,Homo sapiens (Human),9606,130,90,36


first raw record keys:
['_id', 'acc', 'curated-lip-elm', 'curated-lip-merge', 'curated-lip-priority', 'gene', 'homology-domain-gene3d', 'homology-domain-merge', 'homology-domain-pfam', 'homology-msa_entropy-psiblast', 'homology-msa_information_content-psiblast', 'homology-msa_occupancy-psiblast', 'length', 'localization', 'msa_consensus', 'ncbi_taxon_id', 'organism', 'prediction-coil-fess', 'prediction-disorder-dis465', 'prediction-disorder-disHL', 'prediction-disorder-espD', 'prediction-disorder-espN', 'prediction-disorder-espX', 'prediction-disorder-glo', 'prediction-disorder-iupl', 'prediction-disorder-iups', 'prediction-disorder-mobidb_lite', 'prediction-disorder-priority', 'prediction-disorder-th_50', 'prediction-disorder-vsl', 'prediction-helix-fess', 'prediction-lip-anchor', 'prediction-lip-priority', 'prediction-low_complexity-merge', 'prediction-low_complexity-mobidb_lite_sub', 'prediction-low_complexity-pfilt', 'prediction-low_complexity-seg', 'prediction-negative_polyelectro

In [32]:
def summarize_mobidb_block(value):
    if not isinstance(value, dict):
        return value

    summary = {}
    for key, item in value.items():
        if key == "scores" and isinstance(item, list):
            summary["scores_len"] = len(item)
            summary["scores_head"] = item[:5]
        else:
            summary[key] = item
    return summary


if raw_records:
    first = raw_records[0]
    sequence = first.get("sequence") or ""
    annotation_keys = [
        key
        for key, value in first.items()
        if isinstance(value, dict) and ("regions" in value or "scores" in value)
    ]

    core_preview = {
        "_id": first.get("_id"),
        "acc": first.get("acc"),
        "length": first.get("length"),
        "organism": first.get("organism"),
        "ncbi_taxon_id": first.get("ncbi_taxon_id"),
        "sequence_head": sequence[:80],
        "sequence_length": len(sequence),
    }
    annotation_preview = {
        key: summarize_mobidb_block(first[key])
        for key in annotation_keys[:20]
    }

    print("Core fields from first raw record:")
    print(json.dumps(core_preview, ensure_ascii=False, indent=2))
    print("\nFirst annotation blocks with regions/scores shortened:")
    print(json.dumps(annotation_preview, ensure_ascii=False, indent=2))
else:
    print("Run the previous raw preview cell after downloading the raw archive.")

Core fields from first raw record:
{
  "_id": {
    "$oid": "6329714cae6758aa816b7176"
  },
  "acc": "W6PPR4",
  "length": 4332,
  "organism": "Mus musculus (Mouse)",
  "ncbi_taxon_id": 10090,
  "sequence_head": "MAHAASQLKKNRDLEINAEEETEKKRKHRKRSRDRKKKSDANASYLRAARAGHLEKALDYIKNGVDVNICNQNGLNALHL",
  "sequence_length": 4332
}

First annotation blocks with regions/scores shortened:
{
  "curated-lip-elm": {
    "regions": [
      [
        1987,
        1996
      ]
    ],
    "content_fraction": 0.002,
    "content_count": 10
  },
  "curated-lip-merge": {
    "regions": [
      [
        1987,
        1996
      ]
    ],
    "content_fraction": 0.002,
    "content_count": 10
  },
  "homology-msa_entropy-psiblast": {
    "scores_len": 12996,
    "scores_head": [
      0.16224192768870335,
      0.1658096341244076,
      0.1965777260057801,
      0.17708522315803943,
      0.1725573390848306
    ]
  },
  "homology-msa_information_content-psiblast": {
    "scores_len": 12996,
    "scores_head"

In [33]:
SAMPLE_SIZE = 100000

records = list(itertools.islice(iter_mobidb_records(), SAMPLE_SIZE))
print("sample rows:", len(records))
print("first keys:", sorted(records[0].keys()) if records else [])

opening processed JSON
sample rows: 100000
first keys: ['Uniprot_ID', 'disorder_content_count', 'disorder_content_fraction', 'disorder_mask', 'disorder_masks', 'disorder_regions', 'disorder_source', 'disorder_sources', 'length', 'mobidb_id', 'organism', 'primary_disorder_variant', 'sequence', 'taxonomy_id']


In [34]:
df = pd.DataFrame(records)
df.head()

,Uniprot_ID,disorder_content_count,disorder_content_fraction,disorder_mask,disorder_masks,disorder_regions,disorder_source,disorder_sources,length,mobidb_id,organism,primary_disorder_variant,sequence,taxonomy_id
0,W6PPR4,1556,0.359187,1111111111111111111111111111111111111111111100...,"{'all_priority': {'content_count': 1556, 'cont...","[[1, 44], [862, 881], [1510, 1537], [1967, 199...","curated-disorder-merge,homology-disorder-merge...",[prediction-disorder-priority],4332,6329714cae6758aa816b7176,Mus musculus (Mouse),all_priority,MAHAASQLKKNRDLEINAEEETEKKRKHRKRSRDRKKKSDANASYL...,10090
1,Q14596,618,0.639752,0000000000000000000000000000000000000000000000...,"{'all_priority': {'content_count': 618, 'conte...","[[86, 159], [169, 213], [256, 302], [314, 364]...","curated-disorder-merge,homology-disorder-merge...",[prediction-disorder-priority],966,6329714cae6758aa816b7177,Homo sapiens (Human),all_priority,MEPQVTLNVTFKNEIQSFLVSDPENTTWADIEAMVKVSFDLNTIQI...,9606
2,Q96CV9,321,0.556326,1111111111111111111111111111100000000000000000...,"{'all_priority': {'content_count': 321, 'conte...","[[1, 29], [102, 146], [154, 230], [234, 244], ...","curated-disorder-merge,homology-disorder-merge...","[curated-disorder-merge, prediction-disorder-p...",577,6329714cae6758aa816b7178,Homo sapiens (Human),all_priority,MSHQPLSCLTEKEDSPSESTGNGPPHLAHPNLDTFTPEELLQQMKE...,9606
3,P40458,385,0.727788,1111111111111111111111111111111111111111111111...,"{'all_priority': {'content_count': 385, 'conte...","[[1, 215], [342, 399], [413, 485], [491, 529]]","curated-disorder-merge,homology-disorder-merge...","[curated-disorder-merge, prediction-disorder-p...",529,6329714cae6758aa816b7179,Saccharomyces cerevisiae (strain ATCC 204508 /...,all_priority,MVLEYQQREGKGSSSKSMPPDSSSTTIHTCSEAQTGEDKGLLDPHL...,559292
4,Q8MQJ7,394,0.460819,0000000000000000000000000000000000000000000000...,"{'all_priority': {'content_count': 394, 'conte...","[[276, 633], [670, 694], [845, 855]]","curated-disorder-merge,homology-disorder-merge...",[prediction-disorder-priority],855,6329714cae6758aa816b717a,Drosophila melanogaster (Fruit fly),all_priority,MNIVGEYEYSSKDMLGHGAFAVVYKGRHRKKHMPVAIKCITKKGQL...,7227


In [41]:
df['disorder_masks'].iloc[0].keys()

dict_keys(['all_priority', 'curated', 'homology', 'prediction_mobidb_lite', 'prediction_priority'])

In [52]:
(df['disorder_content_fraction'] > 0.40).sum()

11306

In [53]:
df

,Uniprot_ID,disorder_content_count,disorder_content_fraction,disorder_mask,disorder_masks,disorder_regions,disorder_source,disorder_sources,length,mobidb_id,organism,primary_disorder_variant,sequence,taxonomy_id
0,W6PPR4,1556,0.359187,1111111111111111111111111111111111111111111100...,"{'all_priority': {'content_count': 1556, 'cont...","[[1, 44], [862, 881], [1510, 1537], [1967, 199...","curated-disorder-merge,homology-disorder-merge...",[prediction-disorder-priority],4332,6329714cae6758aa816b7176,Mus musculus (Mouse),all_priority,MAHAASQLKKNRDLEINAEEETEKKRKHRKRSRDRKKKSDANASYL...,10090
1,Q14596,618,0.639752,0000000000000000000000000000000000000000000000...,"{'all_priority': {'content_count': 618, 'conte...","[[86, 159], [169, 213], [256, 302], [314, 364]...","curated-disorder-merge,homology-disorder-merge...",[prediction-disorder-priority],966,6329714cae6758aa816b7177,Homo sapiens (Human),all_priority,MEPQVTLNVTFKNEIQSFLVSDPENTTWADIEAMVKVSFDLNTIQI...,9606
2,Q96CV9,321,0.556326,1111111111111111111111111111100000000000000000...,"{'all_priority': {'content_count': 321, 'conte...","[[1, 29], [102, 146], [154, 230], [234, 244], ...","curated-disorder-merge,homology-disorder-merge...","[curated-disorder-merge, prediction-disorder-p...",577,6329714cae6758aa816b7178,Homo sapiens (Human),all_priority,MSHQPLSCLTEKEDSPSESTGNGPPHLAHPNLDTFTPEELLQQMKE...,9606
3,P40458,385,0.727788,1111111111111111111111111111111111111111111111...,"{'all_priority': {'content_count': 385, 'conte...","[[1, 215], [342, 399], [413, 485], [491, 529]]","curated-disorder-merge,homology-disorder-merge...","[curated-disorder-merge, prediction-disorder-p...",529,6329714cae6758aa816b7179,Saccharomyces cerevisiae (strain ATCC 204508 /...,all_priority,MVLEYQQREGKGSSSKSMPPDSSSTTIHTCSEAQTGEDKGLLDPHL...,559292
4,Q8MQJ7,394,0.460819,0000000000000000000000000000000000000000000000...,"{'all_priority': {'content_count': 394, 'conte...","[[276, 633], [670, 694], [845, 855]]","curated-disorder-merge,homology-disorder-merge...",[prediction-disorder-priority],855,6329714cae6758aa816b717a,Drosophila melanogaster (Fruit fly),all_priority,MNIVGEYEYSSKDMLGHGAFAVVYKGRHRKKHMPVAIKCITKKGQL...,7227
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,A0A1X3LDC9,11,0.040293,0000000000000000000000000000000000000000000000...,"{'all_priority': {'content_count': 11, 'conten...","[[263, 273]]","curated-disorder-merge,homology-disorder-merge...",[homology-disorder-merge],273,632976f1ae6758aa816d04a4,Escherichia coli TA054,all_priority,MSCEELEIVWNNIKAEARTLADCEPMLASFYHATLLKHENLGSALS...,656433
99996,A0A1X3LDF3,13,0.083871,0000000000000000000000000000000000000000000000...,"{'all_priority': {'content_count': 13, 'conten...","[[143, 155]]","curated-disorder-merge,homology-disorder-merge...",[homology-disorder-merge],155,632976f1ae6758aa816d04a5,Escherichia coli TA054,all_priority,MSEQNNTEMTFQIQRIYTKDISFEAPNAPHVFQKDWQPEVKLDLDT...,656433
99997,A0A1X3LDF6,16,0.105263,0000000000000000000000000000000000000000000000...,"{'all_priority': {'content_count': 16, 'conten...","[[137, 152]]","curated-disorder-merge,homology-disorder-merge...",[homology-disorder-merge],152,632976f1ae6758aa816d04a6,Escherichia coli TA054,all_priority,MMKKIDVKILDPRVGKEFPLPTYATSGSAGLDLRACLDDAVELAPG...,656433
99998,A0A1X3LDK8,0,0.000000,0000000000000000000000000000000000000000000000...,"{'all_priority': {'content_count': 0, 'content...",[],"curated-disorder-merge,homology-disorder-merge...",[],198,632976f1ae6758aa816d04a7,Escherichia coli TA054,all_priority,MAEKQTAKRNRREEILQSLALMLESSDGSQRITTAKLAASVGVSEA...,656433


In [48]:
df['disorder_masks'].iloc[0]['prediction_priority']

{'content_count': 1556,
 'content_fraction': 0.359187,
 'mask': '11111111111111111111111111111111111111111111000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000011111111111111111111000000000000000000000000000000000000000000000000000000

In [ ]:
df[df['Uniprot_ID'] == 'Q6TL19']

In [ ]:
df['disorder_regions'].value_counts()

In [ ]:
if {"Uniprot_ID", "sequence", "disorder_mask"}.issubset(df.columns):
    sequence_lengths = df["sequence"].fillna("").map(len)
    mask_lengths = df["disorder_mask"].fillna("").map(len)
    disorder_percents = df["disorder_mask"].fillna("").map(
        lambda mask: mask.count("1") / len(mask) * 100 if mask else 0
    )

    print("bad sequence/mask lengths:", int((sequence_lengths != mask_lengths).sum()))
    print("sequence length min/max/avg:", int(sequence_lengths.min()), int(sequence_lengths.max()), f"{sequence_lengths.mean():.2f}")
    print("disorder percent min/max/avg:", f"{disorder_percents.min():.2f}", f"{disorder_percents.max():.2f}", f"{disorder_percents.mean():.2f}")
else:
    print("This looks like the raw/full MobiDB schema, not the compact dataset schema.")

In [ ]:
if "disorder_masks" in df.columns:
    variant_rows = []
    for variant_name in sorted({name for item in df["disorder_masks"].dropna() for name in item.keys()}):
        counts = []
        fractions = []
        nonempty = 0
        for item in df["disorder_masks"].dropna():
            variant = item.get(variant_name, {})
            mask = variant.get("mask", "")
            count = mask.count("1")
            counts.append(count)
            fractions.append(count / len(mask) * 100 if mask else 0)
            nonempty += count > 0

        variant_rows.append(
            {
                "variant": variant_name,
                "records_with_disorder": nonempty,
                "records": len(counts),
                "avg_disorder_percent": sum(fractions) / len(fractions) if fractions else 0,
                "max_disorder_percent": max(fractions) if fractions else 0,
            }
        )

    display(pd.DataFrame(variant_rows).sort_values("records_with_disorder", ascending=False))
else:
    print("No disorder_masks column yet. Rebuild the MobiDB JSON with the updated pipeline.")

In [ ]:
if {"Uniprot_ID", "organism", "taxonomy_id", "sequence", "disorder_mask"}.issubset(df.columns):
    view = df.assign(
        sequence_length=df["sequence"].fillna("").map(len),
        disorder_percent=df["disorder_mask"].fillna("").map(
            lambda mask: mask.count("1") / len(mask) * 100 if mask else 0
        ),
    )
    view[["Uniprot_ID", "organism", "taxonomy_id", "sequence_length", "disorder_percent"]].sort_values(
        "disorder_percent", ascending=False
    ).head(10)
else:
    raw_region_keys = [
        key
        for key, value in records[0].items()
        if isinstance(value, dict) and "regions" in value
    ] if records else []
    print("region annotation keys in first raw record:")
    print(raw_region_keys[:50])

In [ ]:
organism_column = "organism" if "organism" in df.columns else None
if organism_column:
    for organism, count in Counter(df[organism_column].fillna("")).most_common(10):
        print(count, organism, sep="\t")